# CR Impact Analyzer
## Notebook 5 — Hyperparameter Evaluation

### Step 1 — Install evaluation dependencies

In [ ]:
import subprocess
subprocess.run(["pip", "install", "rouge-score", "pandas", "matplotlib",
                "seaborn", "tqdm"], capture_output=True)
print(" Dependencies ready")

### Step 2 — Define the parameter grid

In [ ]:
import os

BASE_DIR         = os.getcwd()
PDF_DIR          = os.path.join(BASE_DIR, "pdfs")
OLLAMA_BASE_URL  = "http://localhost:11434"
COLLECTION_NAME  = "system_docs"


PARAM_GRID = {

    "embedding_models": [
        "BAAI/bge-m3",                      
        "all-MiniLM-L6-v2",                        # fast baseline
        "all-mpnet-base-v2",                        # quality baseline
        "paraphrase-multilingual-MiniLM-L12-v2",   # Arabic+English light
        "paraphrase-multilingual-mpnet-base-v2",   # Arabic+English best
        "multi-qa-mpnet-base-dot-v1",              # Q&A specialized
    ],

    "chunk_sizes": [
        500,    # small
        800,    # medium (current best)
        1200,   # large
    ],
    "chunk_overlaps": [
        100,   # ~12-20% of chunk
        200,   # ~16-25% of chunk
    ],
    "top_k_targeted": [
        4,    # focused
        6,    # balanced
        8,    # comprehensive
    ],

    "top_k_general": [
        0,    # no cross-CR context
        2,    # minimal
        4,    # moderate
    ],

    "llm_models": [
        "llama3",
        "mistral",
        "gemma2:9b",
        "mistral-nemo",
        "phi4:latest",
         "qwen2.5:14b",   
         "gemma:7b",
    ],
}


from itertools import product

embedding_chunk_combos = len(PARAM_GRID["embedding_models"]) * \
                         len(PARAM_GRID["chunk_sizes"]) * \
                         len(PARAM_GRID["chunk_overlaps"])

retrieval_combos = len(PARAM_GRID["top_k_targeted"]) * \
                   len(PARAM_GRID["top_k_general"])

total_runs = embedding_chunk_combos * retrieval_combos * len(PARAM_GRID["llm_models"])

print(f" Parameter grid defined")
print(f"   Embedding × Chunk combinations : {embedding_chunk_combos}")
print(f"   Retrieval combinations          : {retrieval_combos}")
print(f"   LLM models                      : {len(PARAM_GRID['llm_models'])}")
print(f"   Total runs                      : {total_runs}")
print(f"   Estimated time                  : ~{total_runs * 2}-{total_runs * 4} minutes")

### Step 3 — Define evaluation dataset

Use your real CRs here. The more CRs you add, the more reliable the evaluation.


In [ ]:
EVAL_DATASET = [

    {
        "id":        "CR_ID",
        "cr_number": "CR_Number",
        "cr": (
            " "
        ),
        "reference": (
            " "
        )
    },

    {
        "id":        "CR_ID",
        "cr_number": "CR_Number",
        "cr": (
            " "
        ),
        "reference": (
            " "
        )
    },

    {
        "id":        "CR_ID",
        "cr_number": "CR_Number",
        "cr": (
            " "
        ),
        "reference": (
            " "
        )
    },

    {
        "id":        "CR_ID",
        "cr_number": "CR_Number",
        "cr": (
            " "
        ),
        "reference": (
            " "
        )
    },
]


###  Step 4 — Helper functions

In [ ]:
import re
import time
import json
import requests
import chromadb
import pymupdf as fitz
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def extract_cr_number(filename):
    match = re.search(r'(CR|PM)-?\d{4}-?\d+', filename, re.IGNORECASE)
    return match.group(0).upper() if match else "UNKNOWN"

def clean_text(text):
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {3,}', ' ', text)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    return text.strip()

def extract_all_pdfs(pdf_dir):
    pdf_files = [f for f in os.listdir(pdf_dir) if f.lower().endswith(".pdf")]
    docs = []
    for fname in sorted(pdf_files):
        path = os.path.join(pdf_dir, fname)
        doc  = fitz.open(path)
        pages = []
        for i, page in enumerate(doc, 1):
            t = page.get_text("text")
            if t.strip():
                pages.append(f"[Page {i}]\n{t}")
        doc.close()
        text = clean_text("\n\n".join(pages))
        if text:
            docs.append({"source": fname, "cr_number": extract_cr_number(fname), "text": text})
    return docs


def build_vector_db(docs, embed_model, chunk_size, chunk_overlap, store_path):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    all_chunks = []
    for doc in docs:
        for i, chunk in enumerate(splitter.split_text(doc["text"])):
            all_chunks.append({
                "source":      doc["source"],
                "cr_number":   doc["cr_number"],
                "chunk_index": i,
                "text":        chunk
            })

    os.makedirs(store_path, exist_ok=True)
    client = chromadb.PersistentClient(path=store_path)
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

    BATCH = 64
    for start in range(0, len(all_chunks), BATCH):
        batch      = all_chunks[start:start+BATCH]
        texts      = [c["text"] for c in batch]
        metadatas  = [{"source": c["source"], "cr_number": c["cr_number"], "chunk_index": c["chunk_index"]} for c in batch]
        ids        = [f"chunk_{start+i}" for i in range(len(batch))]
        embeddings = embed_model.encode(texts, show_progress_bar=False).tolist()
        collection.add(embeddings=embeddings, documents=texts, metadatas=metadatas, ids=ids)

    return collection, len(all_chunks)


def retrieve(collection, embed_model, query, cr_number, top_k_targeted, top_k_general):
    q_emb = embed_model.encode([query]).tolist()

    # Targeted
    targeted = []
    try:
        r = collection.query(query_embeddings=q_emb, n_results=top_k_targeted,
                             where={"cr_number": cr_number},
                             include=["documents", "metadatas", "distances"])
        targeted = [{"text": d, "source": m["source"], "cr_number": m["cr_number"],
                     "relevance": round(1-dist, 4)}
                    for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]
    except Exception:
        pass

    # General
    general = []
    if top_k_general > 0:
        r = collection.query(query_embeddings=q_emb, n_results=top_k_general+10,
                             include=["documents", "metadatas", "distances"])
        for d, m, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0]):
            if m.get("cr_number") != cr_number:
                general.append({"text": d, "source": m["source"], "cr_number": m["cr_number"],
                                "relevance": round(1-dist, 4)})
            if len(general) >= top_k_general:
                break

    hits = sorted(targeted + general, key=lambda x: x["relevance"], reverse=True)
    avg_relevance = round(sum(h["relevance"] for h in hits) / max(len(hits), 1), 4)
    return hits, avg_relevance


PROMPT_TEMPLATE = """You are a senior OMS software architect.
Using ONLY the documentation below, analyze this Change Request.

CHANGE REQUEST: {cr}

DOCUMENTATION:
{context}

Provide analysis covering:
1. IMPACT ANALYSIS - affected components, risk level, complexity
2. DEPENDENCIES - internal, external, API, configuration dependencies
3. REQUIRED ACTIONS - implementation steps, testing scope
4. RISKS & MITIGATION - top risks and mitigation strategies
"""

def call_llm(cr, context, model):
    prompt  = PROMPT_TEMPLATE.format(cr=cr, context=context)
    payload = {"model": model, "prompt": prompt, "stream": False}
    t0 = time.time()
    try:
        resp = requests.post(f"{OLLAMA_BASE_URL}/api/generate",
                             json=payload, timeout=600)
        resp.raise_for_status()
        answer   = resp.json().get("response", "")
        duration = round(time.time() - t0, 1)
        return answer, duration, None
    except Exception as e:
        return "", 0, str(e)


def score_faithfulness(answer, context):
    sentences    = [s.strip() for s in answer.split(".") if len(s.strip()) > 20]
    if not sentences: return 0.0
    context_lower = context.lower()
    supported = sum(
        1 for s in sentences
        if sum(1 for w in s.split() if len(w)>4 and w.lower() in context_lower)
           / max(len([w for w in s.split() if len(w)>4]), 1) >= 0.3
    )
    return round(supported / len(sentences), 4)

def score_relevancy(cr, answer):
    cr_words = set(w.lower() for w in cr.split() if len(w) > 3)
    return round(sum(1 for w in cr_words if w in answer.lower()) / max(len(cr_words), 1), 4)

def score_structure(answer):
    sections = ["impact", "dependenc", "action", "risk"]
    return round(sum(1 for s in sections if s in answer.lower()) / len(sections), 4)

def score_rouge(answer, reference):
    return round(rouge.score(answer, reference)["rougeL"].fmeasure, 4)

def score_semantic(answer, reference, embed_model):
    import numpy as np
    e1 = embed_model.encode([answer])[0]
    e2 = embed_model.encode([reference])[0]
    cos = np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2))
    return round(float(cos), 4)


print(" Helper functions ready")

### Step 5 — Build all vector databases

One DB is built per (embedding_model × chunk_size × chunk_overlap) combination.

In [ ]:
import itertools

# Extract all PDFs once
print("Extracting PDFs...")
docs = extract_all_pdfs(PDF_DIR)
print(f" {len(docs)} PDFs loaded\n")

# Build one DB per embedding+chunk combination
db_cache = {}   # key → (collection, embed_model_instance)
embed_model_cache = {}

combos = list(itertools.product(
    PARAM_GRID["embedding_models"],
    PARAM_GRID["chunk_sizes"],
    PARAM_GRID["chunk_overlaps"]
))

print(f"Building {len(combos)} vector databases...\n")

for emb, cs, co in combos:
    key        = f"{emb}__cs{cs}__co{co}"
    store_path = os.path.join(BASE_DIR, "eval_stores", key)

    print(f"   {key}")
    t0 = time.time()

    # Load embedding model (cache to avoid reloading same model)
    if emb not in embed_model_cache:
        print(f"       Loading embedding model: {emb}...")
        embed_model_cache[emb] = SentenceTransformer(emb)

    embed_model = embed_model_cache[emb]
    collection, n_chunks = build_vector_db(docs, embed_model, cs, co, store_path)
    db_cache[key] = (collection, embed_model)

    print(f"        {n_chunks} chunks  ({time.time()-t0:.1f}s)")

print(f"\n  All {len(combos)} vector databases built!")

### Step 6 — Run all combinations

In [ ]:
all_results = []
run_number  = 0

total_runs = (
    len(combos) *
    len(PARAM_GRID["top_k_targeted"]) *
    len(PARAM_GRID["top_k_general"]) *
    len(PARAM_GRID["llm_models"]) *
    len(EVAL_DATASET)
)

print(f"Starting {total_runs} total runs...\n")
print("=" * 70)

for emb, cs, co in combos:
    db_key      = f"{emb}__cs{cs}__co{co}"
    collection, embed_model = db_cache[db_key]

    for tk_t, tk_g in itertools.product(
        PARAM_GRID["top_k_targeted"],
        PARAM_GRID["top_k_general"]
    ):
        for llm in PARAM_GRID["llm_models"]:
            for item in EVAL_DATASET:
                run_number += 1
                config_label = f"emb={emb.split('-')[0]}|cs={cs}|co={co}|tkt={tk_t}|tkg={tk_g}|llm={llm}"
                print(f"[{run_number}/{total_runs}] {config_label} | CR={item['id']}")

                # Retrieve
                hits, avg_relevance = retrieve(
                    collection, embed_model,
                    item["cr"], item["cr_number"],
                    tk_t, tk_g
                )
                context = "\n\n".join([h["text"] for h in hits])

                # Generate
                answer, duration, error = call_llm(item["cr"], context, llm)

                if error:
                    print(f"        LLM error: {error[:80]}")
                    continue

                # Score
                faithfulness = score_faithfulness(answer, context)
                relevancy    = score_relevancy(item["cr"], answer)
                structure    = score_structure(answer)
                rouge_l      = score_rouge(answer, item["reference"])
                semantic     = score_semantic(answer, item["reference"], embed_model)

                # Weighted overall score
                overall = round(
                    faithfulness * 0.30 +
                    relevancy    * 0.25 +
                    structure    * 0.20 +
                    semantic     * 0.15 +
                    rouge_l      * 0.10,
                    4
                )

                all_results.append({
                    "embedding_model": emb,
                    "chunk_size":      cs,
                    "chunk_overlap":   co,
                    "top_k_targeted":  tk_t,
                    "top_k_general":   tk_g,
                    "llm_model":       llm,
                    "cr_id":           item["id"],
                    "avg_retrieval_relevance": avg_relevance,
                    "faithfulness":    faithfulness,
                    "answer_relevancy": relevancy,
                    "structure_score": structure,
                    "rouge_l":         rouge_l,
                    "semantic_score":  semantic,
                    "overall_score":   overall,
                    "duration_sec":    duration,
                    "word_count":      len(answer.split()),
                })
                print(f"        overall={overall:.3f}  faith={faithfulness:.2f}  "
                      f"rel={relevancy:.2f}  struct={structure:.2f}  "
                      f"rouge={rouge_l:.2f}  sem={semantic:.2f}  {duration}s")

print(f"\n All {len(all_results)} runs complete!")

###  Step 7 — Results analysis

In [ ]:
import pandas as pd
pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

df = pd.DataFrame(all_results)

group_cols = ["embedding_model", "chunk_size", "chunk_overlap",
              "top_k_targeted", "top_k_general", "llm_model"]

summary = df.groupby(group_cols).agg(
    overall_score          = ("overall_score",            "mean"),
    faithfulness           = ("faithfulness",             "mean"),
    answer_relevancy       = ("answer_relevancy",         "mean"),
    structure_score        = ("structure_score",          "mean"),
    rouge_l                = ("rouge_l",                  "mean"),
    semantic_score         = ("semantic_score",           "mean"),
    avg_retrieval_relevance= ("avg_retrieval_relevance",  "mean"),
    avg_speed_sec          = ("duration_sec",             "mean"),
).reset_index().sort_values("overall_score", ascending=False)

print("TOP 10 CONFIGURATIONS")
print("=" * 80)
display(summary.head(10))

###  Step 8 — Best configuration

In [ ]:
best = summary.iloc[0]

print("\n" + "═" * 60)
print("   BEST CONFIGURATION FOUND")
print("═" * 60)
print(f"  Embedding model   : {best['embedding_model']}")
print(f"  Chunk size        : {best['chunk_size']}")
print(f"  Chunk overlap     : {best['chunk_overlap']}")
print(f"  Top-K targeted    : {best['top_k_targeted']}")
print(f"  Top-K general     : {best['top_k_general']}")
print(f"  LLM model         : {best['llm_model']}")
print("─" * 60)
print(f"  Overall score     : {best['overall_score']:.3f}")
print(f"  Faithfulness      : {best['faithfulness']:.3f}")
print(f"  Answer relevancy  : {best['answer_relevancy']:.3f}")
print(f"  Structure score   : {best['structure_score']:.3f}")
print(f"  Semantic score    : {best['semantic_score']:.3f}")
print(f"  ROUGE-L           : {best['rouge_l']:.3f}")
print(f"  Avg speed         : {best['avg_speed_sec']:.1f}s per CR")
print("═" * 60)

print("\n Copy these settings into Notebook 2 and Notebook 3:")
print(f"""
# Notebook 2 — Build Vector DB
EMBEDDING_MODEL  = \"{best['embedding_model']}\"
CHUNK_SIZE       = {int(best['chunk_size'])}
CHUNK_OVERLAP    = {int(best['chunk_overlap'])}

# Notebook 3 — Analyze CR
EMBEDDING_MODEL  = \"{best['embedding_model']}\"
LLM_MODEL        = \"{best['llm_model']}\"
TOP_K_TARGETED   = {int(best['top_k_targeted'])}
TOP_K_GENERAL    = {int(best['top_k_general'])}
""")

### Step 9 — Visualize results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="darkgrid")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Hyperparameter Evaluation Results", fontsize=14, fontweight="bold")

params_to_plot = [
    ("embedding_model",  "Embedding Model"),
    ("chunk_size",       "Chunk Size"),
    ("chunk_overlap",    "Chunk Overlap"),
    ("top_k_targeted",   "Top-K Targeted"),
    ("top_k_general",    "Top-K General"),
    ("llm_model",        "LLM Model"),
]

for ax, (param, label) in zip(axes.flatten(), params_to_plot):
    param_summary = df.groupby(param)["overall_score"].mean().reset_index()
    param_summary = param_summary.sort_values("overall_score", ascending=False)

    colors = sns.color_palette("husl", len(param_summary))
    bars   = ax.bar(
        range(len(param_summary)),
        param_summary["overall_score"],
        color=colors
    )
    ax.set_xticks(range(len(param_summary)))
    ax.set_xticklabels(
        [str(v)[:20] for v in param_summary[param]],
        rotation=20, ha="right", fontsize=8
    )
    ax.set_title(label, fontweight="bold")
    ax.set_ylabel("Avg Overall Score")
    ax.set_ylim(0, 1.0)

    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=8
        )

plt.tight_layout()
chart_path = os.path.join(BASE_DIR, "hyperparameter_evaluation.png")
plt.savefig(chart_path, bbox_inches="tight")
plt.show()
print(f"\n Chart saved to: {chart_path}")

###  Step 10 — Save full results

In [ ]:
import datetime

output_dir = os.path.join(BASE_DIR, "evaluations")
os.makedirs(output_dir, exist_ok=True)
timestamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Full results CSV
csv_path = os.path.join(output_dir, f"hyperparam_results_{timestamp}.csv")
df.to_csv(csv_path, index=False)

# Summary CSV
summary_path = os.path.join(output_dir, f"hyperparam_summary_{timestamp}.csv")
summary.to_csv(summary_path, index=False)

# Text report
report_path = os.path.join(output_dir, f"hyperparam_report_{timestamp}.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("CR IMPACT ANALYZER — HYPERPARAMETER EVALUATION REPORT\n")
    f.write(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write("BEST CONFIGURATION\n" + "="*60 + "\n")
    f.write(f"Embedding model : {best['embedding_model']}\n")
    f.write(f"Chunk size      : {int(best['chunk_size'])}\n")
    f.write(f"Chunk overlap   : {int(best['chunk_overlap'])}\n")
    f.write(f"Top-K targeted  : {int(best['top_k_targeted'])}\n")
    f.write(f"Top-K general   : {int(best['top_k_general'])}\n")
    f.write(f"LLM model       : {best['llm_model']}\n")
    f.write(f"Overall score   : {best['overall_score']:.3f}\n\n")
    f.write("TOP 10 CONFIGURATIONS\n" + "="*60 + "\n")
    f.write(summary.head(10).to_string())

print(f" Results saved:")
print(f"   Full results : {csv_path}")
print(f"   Summary      : {summary_path}")
print(f"   Report       : {report_path}")

---
## 📖 How to read the results

### Score weights (overall score)
| Metric | Weight | What it measures |
|---|---|---|
| Faithfulness | 30% | Answer is grounded in retrieved docs, not hallucinated |
| Answer Relevancy | 25% | Answer addresses the actual CR question |
| Structure Score | 20% | All 4 sections present in the answer |
| Semantic Score | 15% | Meaning similarity to reference answer |
| ROUGE-L | 10% | Text overlap with reference answer |

### What each parameter affects
| Parameter | High value effect | Low value effect |
|---|---|---|
| `chunk_size` | More complete context per chunk | More precise, may cut steps |
| `chunk_overlap` | Better continuity | Less redundancy |
| `top_k_targeted` | More from the specific CR | Less noise from same CR |
| `top_k_general` | More cross-CR context | Less cross-CR noise |
| `embedding_model` | Better = higher retrieval relevance | Faster but less accurate |